In [1]:
import scanpy as sc
import anndata as ad
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, rgb2hex
from matplotlib_scalebar.scalebar import ScaleBar
import numpy as np
import warnings
import pandas as pd
warnings.filterwarnings('ignore')
import numpy as np
from sklearn.metrics import jaccard_score
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['pdf.fonttype'] = 42 # ADOBE AI 字帖

from matplotlib.font_manager import fontManager, FontProperties

fontManager.addfont('/data/work/Arial.ttf')

font = FontProperties(fname='/data/work/Arial.ttf')
font_name = font.get_name()
plt.rcParams['font.family'] = font_name

In [2]:
adata = sc.read_h5ad('/data/work/05.cluster/FuseMap/0106/Hippocampus_latent_embeddings_all_single_pretrain/dmt_leiden_20250108_1.h5ad')
names = [
    'B03607C4E6_WT2024071214941.h5ad',
    
    '12_B03605F3G5_WT202403310048.h5ad',
    '13_B03612A1C3_WT202403310056.h5ad',
    '14_A03591A1C3_WT202403310045.h5ad',
    '16_A03592A4C6_WT202403310044.h5ad',
    '18_B03602C4D6_WT202405020031.h5ad',
    '20_B03606F3G5_WT202405020032.h5ad',
    '22_B03606C4E6_WT202403310050.h5ad',
    '23_B03609A4D6_WT202404150263.h5ad',
    '27_B03610C1E3_WT202403310051.h5ad',
    '31_B03619A1D3_WT202403310052.h5ad',
    '35_B03619E4G6_WT202403310053.h5ad',
    '39_A03589A1D4_WT202403310046.h5ad',
    '43_A03590E1G4_WT202403310064.h5ad',
    '47_A03593C1F3_WT202403310068.h5ad',
]

In [3]:
dic_dmt_leiden = {
    '0': 'hip_sc_01',
    '7': 'hip_sc_02',
    
    '1': 'hip_sc_03',
    '3': 'hip_sc_04',
    '6': 'hip_sc_04',
    '14': 'hip_sc_05',
    
    
    '2': 'hip_sc_06',
    '13': 'hip_sc_07',
    '19': 'hip_sc_08',
    '20': 'hip_sc_09',
    
    '4': 'hip_sc_10',
    '21': 'hip_sc_10',
    
    '8': 'hip_sc_11',
    '10': 'hip_sc_12',
    '22': 'hip_sc_13',
    
    '9': 'hip_sc_14',
    '11': 'hip_sc_14',
    '16': 'hip_sc_15',
    '18': 'hip_sc_15',
    '26': 'hip_sc_15',
    '31': 'hip_sc_15',
    
    '12': 'hip_sc_16',
    '15': 'hip_sc_17',
    '25': 'hip_sc_17',
    '29': 'hip_sc_17',
    '17': 'hip_sc_18',
    
    '24': 'z_delete',
    
    
    '5': 'hip_sc_19',
    '23': 'hip_sc_20',
    '27': 'hip_sc_21',
    '28': 'hip_sc_22',
    '30': 'hip_sc_23',
}
adata.obs['dmt_leiden_anno'] = [dic_dmt_leiden[i] for i in adata.obs['dmt_leiden']]
adata = adata[adata.obs['dmt_leiden_anno'] != 'z_delete']
colormap = {
  'hip_sc_01' : '#9b38e9',
  'hip_sc_02' : '#a89630',
  'hip_sc_03': '#5b798b',
  'hip_sc_04' : '#cb2505',
  'hip_sc_05' : '#62e7dd',
  'hip_sc_06' : '#245200',
  'hip_sc_07' : '#374898',
  'hip_sc_08' : '#6d85c7',
  'hip_sc_09' : '#35c498',
  'hip_sc_10' : '#9e2dc6',
  'hip_sc_11' : '#2d7476',
  'hip_sc_12' : '#cb0d6c',
  'hip_sc_13' : '#20ea38',
  'hip_sc_14' : '#0fabb6',
  'hip_sc_15' : '#a59099',
  'hip_sc_16' : '#2bea3a',
  'hip_sc_17' : '#17b064',
  'hip_sc_18' : '#52b8d5',
  'hip_sc_19' : '#da2ef2',
  'hip_sc_20' : '#6240f7',
  'hip_sc_21' : '#c47233',
  'hip_sc_22':'#a83b23',
  'hip_sc_23':'#9994da'
}

In [4]:
adatas = []
for i in set(adata.obs['slice_code']):
    temp = adata[adata.obs['slice_code'] == i].copy()
    sc.pp.normalize_total(temp)
    sc.pp.log1p(temp)
    sc.pp.scale(temp, zero_center=False, max_value=10)
    adatas.append(temp)
adata = ad.concat(adatas)

In [12]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

x_min, x_max, y_min, y_max = float('inf'), float('-inf'), float('inf'), float('-inf')
for name in names:
    adata_temp = adata[adata.obs['slice_code'] == name].copy()
    adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
    adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
    
    x_min = min(x_min, adata_temp.obsm['align_spatial_2d'][:, 0].min())
    x_max = max(x_max, adata_temp.obsm['align_spatial_2d'][:, 0].max())
    y_min = min(y_min, adata_temp.obsm['align_spatial_2d'][:, 1].min())
    y_max = max(y_max, adata_temp.obsm['align_spatial_2d'][:, 1].max())

In [15]:

var = ['CRYAB', 'SLC1A3', 'ZFP36L1', 'NNAT', 'BHLHE22',  
    'TTR', 
    'NFIA', 'TCF4', 'DCX', 'RTN4', 'ZBTB20',  'EOMES','PAX6', 'NTS','NNAT','ZBTB20','BCL11A','BCL11B','SATB1','SATB2','CRYAB',]
import os
for gene in var:
    if os.path.exists(f'/data/work/05.cluster/FuseMap/0116/hip_single/gene_plot/{gene}.png'):
        continue 
    fig = plt.figure(figsize=(64, 6))
    gs = GridSpec(1, 15, figure=fig)
    count = 0
    for name in names:
        adata_temp = adata[adata.obs['slice_code'] == name].copy()
        adata_temp.obsm['align_spatial_2d'] = adata_temp.obsm['align_spatial_2d'] - adata_temp.obsm['align_spatial_2d'].max(axis = 0)
        adata_temp.obsm['align_spatial_2d'][:, 1] = -adata_temp.obsm['align_spatial_2d'][:, 1]
        col = count % 15
        ax = fig.add_subplot(gs[0, col])

        # 绘制嵌入图
        sc.pl.embedding(
            adata_temp, basis="align_spatial_2d", color=gene,
            show=False, s=1, title='', legend_loc=None, ax=ax, cmap = 'Reds',
        )

        # 设置统一的坐标轴范围
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)

        # 关闭坐标轴并设置等比例缩放
        ax.axis('off')
        ax.set_aspect('equal')
        # scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
        # ax.add_artist(scalebar)
        if count == 0:
            scalebar = ScaleBar(0.0097, "mm", fixed_value=1, location = 'lower left', frameon = False,)
            ax.add_artist(scalebar)
        count += 1
    # plt.show()
    plt.savefig(f'/data/work/05.cluster/FuseMap/0116/hip_single/gene_plot/{gene}.png', dpi = 600, bbox_inches = 'tight')
    plt.close()
    # break